In [1]:
import serial
import time
from datetime import datetime

# =========================================================
# CẤU HÌNH KẾT NỐI
# =========================================================
PORT = "COM14"
BAUD_RATE = 19200

LOG_FILE = f"rockblock_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"


# =========================================================
# LOG
# =========================================================
def write_log(message):
    """Ghi message ra màn hình và file log."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"

    print(log_message)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_message + "\n")


# =========================================================
# GỬI AT COMMAND
# =========================================================
def send_cmd(ser, cmd, wait=2):
    """Gửi lệnh AT và ghi command + response vào log."""

    write_log(f">>> {cmd}")

    # Xóa dữ liệu cũ trong buffer
    ser.reset_input_buffer()

    # Gửi command
    ser.write((cmd + "\r").encode("ascii"))

    # Chờ modem phản hồi
    time.sleep(wait)

    # Đọc response
    data = ser.read_all().decode("ascii", errors="ignore").strip()

    if data:
        write_log(f"<<< {data}")
    else:
        write_log("<<< [NO RESPONSE]")

    return data


# =========================================================
# MAIN
# =========================================================
ser = None

try:
    write_log("=" * 70)
    write_log("ROCKBLOCK / IRIDIUM MODEM TEST")
    write_log("=" * 70)

    write_log(f"Serial port : {PORT}")
    write_log(f"Baud rate   : {BAUD_RATE}")
    write_log(f"Log file    : {LOG_FILE}")

    # -----------------------------------------------------
    # 1. KẾT NỐI SERIAL
    # -----------------------------------------------------
    ser = serial.Serial(
        port=PORT,
        baudrate=BAUD_RATE,
        timeout=2
    )

    write_log(f"Connected: {PORT} @ {BAUD_RATE}")

    # -----------------------------------------------------
    # 2. CẤU HÌNH MODEM
    # -----------------------------------------------------
    write_log("")
    write_log("=== INITIAL MODEM CONFIGURATION ===")

    send_cmd(ser, "AT")
    send_cmd(ser, "ATE0")
    send_cmd(ser, "AT&K0")

    # -----------------------------------------------------
    # 3. THÔNG TIN MODEM
    # -----------------------------------------------------
    write_log("")
    write_log("=== MODEM INFORMATION ===")

    firmware = send_cmd(ser, "AT+CGMR")
    imei = send_cmd(ser, "AT+CGSN")
    sim = send_cmd(ser, "AT+CPIN?")

    # -----------------------------------------------------
    # 4. KIỂM TRA SIM
    # -----------------------------------------------------
    if "READY" in sim:
        write_log("[OK] SIM status: READY")
    else:
        write_log("[WARNING] SIM status is not READY")

    # -----------------------------------------------------
    # 5. KIỂM TRA IRIDIUM NETWORK
    # -----------------------------------------------------
    write_log("")
    write_log("=" * 70)
    write_log("BAT DAU KIEM TRA TIN HIEU VA DONG BO IRIDIUM")
    write_log("Yeu cau dat anten ngoai troi")
    write_log("=" * 70)

    checks = 10
    network_connected = False

    for i in range(checks):

        write_log("")
        write_log(f"----- Check {i + 1}/{checks} -----")

        # -------------------------------------------------
        # CSQ
        # -------------------------------------------------
        csq = send_cmd(
            ser,
            "AT+CSQ",
            wait=2
        )

        # -------------------------------------------------
        # MSSTM
        # -------------------------------------------------
        msstm = send_cmd(
            ser,
            "AT-MSSTM",
            wait=3
        )

        # -------------------------------------------------
        # KIỂM TRA NETWORK
        # -------------------------------------------------
        msstm_lower = msstm.lower()

        if "no network service" not in msstm_lower:

            write_log(
                "[OK] Modem da ket noi thanh cong "
                "voi mang Iridium va dong bo thoi gian!"
            )

            network_connected = True
            break

        else:
            write_log(
                "[INFO] Chua co network service. "
                "Tiep tuc cho modem tim mang..."
            )

        # -------------------------------------------------
        # WAIT
        # -------------------------------------------------
        if i < checks - 1:

            write_log("Waiting 5 seconds...")
            time.sleep(5)

    # -----------------------------------------------------
    # 6. KẾT QUẢ CUỐI
    # -----------------------------------------------------
    write_log("")
    write_log("=" * 70)
    write_log("KET QUA")
    write_log("=" * 70)

    if network_connected:
        write_log("[SUCCESS] Iridium network available.")
    else:
        write_log(
            "[FAIL] Khong ket noi duoc Iridium network "
            f"sau {checks} lan kiem tra."
        )

    write_log("")
    write_log(f"Log saved to: {LOG_FILE}")


# =========================================================
# ERROR
# =========================================================
except serial.SerialException as se:

    write_log(f"[SERIAL ERROR] {se}")

except Exception as e:

    write_log(f"[ERROR] {e}")


# =========================================================
# CLOSE SERIAL
# =========================================================
finally:

    if ser is not None and ser.is_open:

        ser.close()

        write_log("Serial closed.")

    write_log("=" * 70)
    write_log("TEST FINISHED")
    write_log("=" * 70)

[2026-10-02 07:28:51] ======================================================================
[2026-10-02 07:28:51] ROCKBLOCK / IRIDIUM MODEM TEST
[2026-10-02 07:28:51] ======================================================================
[2026-10-02 07:28:51] Serial port : COM14
[2026-10-02 07:28:51] Baud rate   : 19200
[2026-10-02 07:28:51] Log file    : rockblock_log_20261002_072851.txt
[2026-10-02 07:28:51] Connected: COM14 @ 19200
[2026-10-02 07:28:51] 
[2026-10-02 07:28:51] === INITIAL MODEM CONFIGURATION ===
[2026-10-02 07:28:51] >>> AT
[2026-10-02 07:28:53] <<< OK
[2026-10-02 07:28:53] >>> ATE0
[2026-10-02 07:28:55] <<< OK
[2026-10-02 07:28:55] >>> AT&K0
[2026-10-02 07:28:57] <<< OK
[2026-10-02 07:28:57] 
[2026-10-02 07:28:57] === MODEM INFORMATION ===
[2026-10-02 07:28:57] >>> AT+CGMR
[2026-10-02 07:28:59] <<< Call Processor Version: TA21004

Modem DSP Version: 1.7 svn: 2358

DBB Version: 0x0001 (ASIC)

RFA Version: 0x001a (GRFA1B)

NVM Version: KVS

Hardware Version: BOOT07d4

In [ ]:
import serial
import time
 
# Cấu hình cổng serial (thay thế 'COM3' bằng cổng kết nối thực tế của bạn trên Windows/Linux/Mac)
SERIAL_PORT = 'COM14'
BAUD_RATE = 19200

def test_rockblock():
    try:
        # Mở kết nối Serial với timeout là 2 giây
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=2)
        time.sleep(1)
        
        print("Đang kết nối với RockBLOCK 9603...")

        # 1. Kiểm tra kết nối AT
        print("\n-> Gửi lệnh kiểm tra: AT")
        ser.write(b'AT\r')
        time.sleep(0.5)
        response = ser.read_all().decode('utf-8', errors='ignore')
        print(f"Phản hồi từ thiết bị:\n{response}")

        # 2. Tắt điều luồng (flow control) cho chế độ 3-wire
        print("\n-> Gửi lệnh: AT&K0")
        ser.write(b'AT&K0\r')
        time.sleep(0.5)
        print(ser.read_all().decode('utf-8', errors='ignore'))

        # 3. Viết thông điệp "Hello World" vào bộ đệm SBD
        print("\n-> Gửi thông điệp: AT+SBDWT Hello World")
        ser.write(b'AT+SBDWT Hello World\r')
        time.sleep(1)
        print(ser.read_all().decode('utf-8', errors='ignore'))

        # 4. Thực hiện phiên truyền/nhận SBD (SBDIX)
        print("\n-> Bắt đầu phiên truyền tin (AT+SBDIX)... Quá trình này có thể mất vài giây.")
        ser.write(b'AT+SBDIX\r')
        
        # Phiên truyền vệ tinh có thể mất nhiều thời gian hơn, chờ khoảng 10 giây để nhận kết quả phản hồi
        time.sleep(10)
        sbdix_response = ser.read_all().decode('utf-8', errors='ignore')
        print(f"Kết quả phiên SBDIX:\n{sbdix_response}")

        ser.close()
        print("\nĐã đóng kết nối Serial.")

    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")

if __name__ == "__main__":
    test_rockblock()

In [2]:
import serial
import time

SERIAL_PORT = 'COM14'
BAUD_RATE = 19200

def send_to_satellite():
    ser = None
    try:
        print(f"Đang mở kết nối cổng {SERIAL_PORT}...")
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=2)
        time.sleep(1)
        
        ser.reset_input_buffer()
        ser.reset_output_buffer()

        # 1. Kiểm tra kết nối cơ bản
        print("\n--- Bước 1: Kiểm tra kết nối thiết bị ---")
        ser.write(b'AT\r')
        time.sleep(0.5)
        response = ser.read_all().decode('utf-8', errors='ignore')
        print(response.strip())
        
        if "OK" not in response:
            print("Không nhận được phản hồi hợp lệ từ RockBLOCK.")
            return

        # 2. Thiết lập chế độ 3-wire
        print("\n--- Bước 2: Thiết lập chế độ 3-wire (AT&K0) ---")
        ser.write(b'AT&K0\r')
        time.sleep(0.5)
        print(ser.read_all().decode('utf-8', errors='ignore').strip())

        # 3. Ghi thông điệp vào bộ đệm MO
        print("\n--- Bước 3: Ghi thông điệp vào bộ đệm thiết bị ---")
        print("Đang gửi: AT+SBDWT Hello World")
        ser.write(b'AT+SBDWT Hello World\r')
        time.sleep(1)
        res_wt = ser.read_all().decode('utf-8', errors='ignore').strip()
        print(res_wt)

        # Nếu bước ghi lỗi, thử cấp lại lệnh hoặc dừng
        if "OK" not in res_wt:
            print("Cảnh báo: Lệnh SBDWT chưa trả về OK, thử tiếp tục phiên SBDIX...")

        # 4. Kích hoạt truyền dữ liệu lên vệ tinh
        print("\n--- Bước 4: Thực hiện truyền dữ liệu lên vệ tinh Iridium ---")
        print("Đang gọi AT+SBDIX (Hãy đảm bảo thiết bị đang hướng ra ngoài trời)...")
        ser.write(b'AT+SBDIX\r')
        
        # Chờ quá trình kết nối vệ tinh (có thể mất 10-20 giây)
        time.sleep(15)
        
        sbdix_response = ser.read_all().decode('utf-8', errors='ignore')
        print(f"Kết quả phản hồi từ phiên SBDIX:\n{sbdix_response.strip()}")

        if "+SBDIX:" in sbdix_response:
            parts = sbdix_response.split("+SBDIX:")[1].strip().split(",")
            mo_status = int(parts[0].strip())
            
            if mo_status == 0:
                print("\nTHÀNH CÔNG: Thông điệp đã được gửi qua vệ tinh thành công!")
            else:
                print(f"\nGỬI KHÔNG THÀNH CÔNG: Mã trạng thái lỗi MO = {mo_status}. (Lỗi 32 nghĩa là không bắt được sóng vệ tinh, hãy kiểm tra lại không gian ngoài trời và anten).")

    except serial.SerialException as e:
        print(f"Lỗi cổng Serial: {e}")
    except Exception as e:
        print(f"Đã xảy ra lỗi không xác định: {e}")
    finally:
        if ser and ser.is_open:
            ser.close()
            print("\nĐã đóng kết nối Serial.")

if __name__ == "__main__":
    send_to_satellite()

Đang mở kết nối cổng COM14...

--- Bước 1: Kiểm tra kết nối thiết bị ---
AT
OK

--- Bước 2: Thiết lập chế độ 3-wire (AT&K0) ---
AT&K0
OK

--- Bước 3: Ghi thông điệp vào bộ đệm thiết bị ---
Đang gửi: AT+SBDWT Hello World
AT+SBDWT Hello World
ERROR
Cảnh báo: Lệnh SBDWT chưa trả về OK, thử tiếp tục phiên SBDIX...

--- Bước 4: Thực hiện truyền dữ liệu lên vệ tinh Iridium ---
Đang gọi AT+SBDIX (Hãy đảm bảo thiết bị đang hướng ra ngoài trời)...
Kết quả phản hồi từ phiên SBDIX:
AT+SBDIX
+SBDIX: 32, 0, 2, 0, 0, 0

OK

GỬI KHÔNG THÀNH CÔNG: Mã trạng thái lỗi MO = 32. (Lỗi 32 nghĩa là không bắt được sóng vệ tinh, hãy kiểm tra lại không gian ngoài trời và anten).

Đã đóng kết nối Serial.


In [1]:
import serial
import time

PORT = "COM16"

BAUDRATES = [
    1200,
    2400,
    4800,
    9600,
    19200,
    38400,
    57600,
    115200,
    230400,
    460800,
    921600
]

print("=" * 70)
print("RFD900x UART ACTIVE BAUDRATE & COMMAND SCANNER")
print("=" * 70)
print(f"PORT = {PORT}")
print("Hãy rút cáp nguồn/USB của modem ra, chạy script này, sau đó cắm lại nguồn cho modem.")
print("-" * 70)

for baud in BAUDRATES:
    print(f"\n--- Đang kiểm tra Baudrate: {baud} ---")
    
    try:
        ser = serial.Serial(
            port=PORT,
            baudrate=baud,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=0.5
        )

        ser.reset_input_buffer()
        ser.reset_output_buffer()

        # 1. Lắng nghe dữ liệu khởi động (Boot message) trong 1.5 giây đầu
        start = time.time()
        boot_data = b""
        while time.time() - start < 1.5:
            rx = ser.read(1024)
            if rx:
                boot_data += rx

        if boot_data:
            print("  [+] Nhận dữ liệu boot/thụ động:")
            print("      ASCII:", repr(boot_data))

        # 2. Thử gửi chuỗi lệnh chuyển chế độ (Guard time + '+++')
        # RFD900 yêu cầu: 1 giây nghỉ -> '+++' -> 1 giây nghỉ
        print("  [*] Thử gửi lệnh vào Command Mode ('+++')...")
        time.sleep(1.1)          # Nghỉ trước khi gửi
        ser.write(b"+++")
        ser.flush()
        time.sleep(1.1)          # Nghỉ sau khi gửi để modem kịp phản hồi

        # 3. Đọc phản hồi sau lệnh '+++'
        cmd_response = ser.read(1024)
        
        if cmd_response:
            print("  [+] Phản hồi lệnh:")
            print("      ASCII:", repr(cmd_response))
            print("      HEX  :", cmd_response.hex(" "))
            
            if b"OK" in cmd_response.upper() or b"ATI" in cmd_response.upper() or b"RFD" in cmd_response.upper():
                print(f"\n>>> TÌM THẤY BAUDRATE CHÍNH XÁC: {baud} <<<")
                ser.close()
                break
        else:
            print("  [-] Không có phản hồi lệnh ở baud này.")

        ser.close()

    except Exception as e:
        print("  [!] LỖI:", e)

print("\n" + "=" * 70)
print("HOÀN TẤT QUÉT")
print("=" * 70)

RFD900x UART ACTIVE BAUDRATE & COMMAND SCANNER
PORT = COM16
Hãy rút cáp nguồn/USB của modem ra, chạy script này, sau đó cắm lại nguồn cho modem.
----------------------------------------------------------------------

--- Đang kiểm tra Baudrate: 1200 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrate: 2400 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrate: 4800 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrate: 9600 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrate: 19200 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrate: 38400 ---
  [*] Thử gửi lệnh vào Command Mode ('+++')...
  [-] Không có phản hồi lệnh ở baud này.

--- Đang kiểm tra Baudrat